# BirdCLEF 2026 — Ensemble: Perch Zero-Shot + Fine-tuned Head

**Strategy:** Blend Perch's built-in species predictions (`label` output) with a fine-tuned MLP head.

- Perch zero-shot uses Perch's pre-trained classifier mapped to 234 target species (LB ~0.825)
- Fine-tuned head adds signal for species not covered by Perch taxonomy
- Weighted blend: `alpha × perch_zero_shot + (1-alpha) × fine_tuned`

Upload `checkpoints/no-human-voice/best_head.weights.h5` as Kaggle dataset `birdclef2026-no-human-voice`.

In [ ]:
# Ensure TF 2.20+ (required for Perch v2 StableHLO compatibility)
import subprocess, sys

def _tf_version():
    try:
        import tensorflow as tf
        return tuple(int(x) for x in tf.__version__.split(".")[:2])
    except Exception:
        return (0, 0)

if _tf_version() < (2, 20):
    import os
    tb_wheel = "/kaggle/input/bc26-tensorflow-2-20-0/wheel/tensorboard-2.20.0-py3-none-any.whl"
    tf_wheel = "/kaggle/input/bc26-tensorflow-2-20-0/wheel/tensorflow-2.20.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl"
    if os.path.isfile(tb_wheel):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", tb_wheel], check=False)
    if os.path.isfile(tf_wheel):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", tf_wheel], check=False)
    else:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tensorflow==2.20.0"], check=False)

import tensorflow as tf
assert tuple(int(x) for x in tf.__version__.split(".")[:2]) >= (2, 20), f"TF 2.20+ required, got {tf.__version__}"
print(f"TensorFlow {tf.__version__}")
tf.experimental.numpy.experimental_enable_numpy_behavior()

In [ ]:
import glob, os, re, time
import h5py
import librosa
import numpy as np
import pandas as pd
import tensorflow as tf
from concurrent.futures import ThreadPoolExecutor

START          = time.time()
TERMINATE_TIME = START + 8.5 * 3600

## Config

In [ ]:
DATA_DIR   = '/kaggle/input/birdclef-2026'
PERCH_DIR  = '/kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/perch_v2_cpu/1'

# Fine-tuned weights (no-human-voice, best model so far)
WEIGHTS_PATH = '/kaggle/input/birdclef2026-no-human-voice/best_head.weights.h5'

# Ensemble blend weight for Perch zero-shot (0.0 = pure fine-tuned, 1.0 = pure Perch)
ALPHA = 0.6    # Perch zero-shot weight

NUM_CLASSES   = 234
HIDDEN_DIM    = 512
DROPOUT       = 0.0    # inference: no dropout
SAMPLE_RATE   = 32_000
CLIP_DURATION = 5
CLIP_SAMPLES  = SAMPLE_RATE * CLIP_DURATION
BATCH_SIZE    = 64

TEST_DIR  = os.path.join(DATA_DIR, 'test_soundscapes')
TRAIN_SC  = os.path.join(DATA_DIR, 'train_soundscapes')
SC_DIR    = TEST_DIR if glob.glob(os.path.join(TEST_DIR, '*.ogg')) else TRAIN_SC
print(f'Soundscapes dir: {SC_DIR}')
print(f'Files: {len(glob.glob(os.path.join(SC_DIR, "*.ogg")))}')

## Perch Species Mapping

In [ ]:
# Load Perch's label taxonomy and map to our 234 target species
bc_labels = pd.read_csv(os.path.join(PERCH_DIR, 'assets/labels.csv'))
bc_labels = (bc_labels.reset_index()
             .rename({'inat2024_fsd50k': 'scientific_name', 'index': 'bc_index'}, axis=1)
             .set_index('scientific_name'))

taxonomy      = pd.read_csv(os.path.join(DATA_DIR, 'taxonomy.csv'))
sample_sub    = pd.read_csv(os.path.join(DATA_DIR, 'sample_submission.csv'))
primary_labels = sample_sub.columns[1:].tolist()

mapping = taxonomy.join(bc_labels, on='scientific_name', how='left')
mapping.bc_index = mapping.bc_index.fillna(len(bc_labels)).astype(int)
mapping = mapping[['primary_label', 'bc_index']].set_index('primary_label')

# Index into Perch label output; unmapped species → last index (will be zero after padding)
perch_indices = [int(mapping.loc[pl][0]) for pl in primary_labels]
n_covered = len(set(perch_indices) - {len(bc_labels)})
print(f'Perch covers {n_covered} / {NUM_CLASSES} target species')

## Load Perch + Fine-tuned Head

In [ ]:
print('Loading Perch …')
perch = tf.saved_model.load(PERCH_DIR)
sig   = perch.signatures['serving_default']

# Probe Perch outputs
_dummy = tf.zeros((1, CLIP_SAMPLES), tf.float32)
_out   = sig(inputs=_dummy)
EMB_KEY = next((k for k in ('embedding','embeddings','label','logits') if k in _out),
               next(iter(_out.keys())))
EMB_DIM = int(_out[EMB_KEY].shape[-1])
print(f'Perch embedding key={EMB_KEY!r} dim={EMB_DIM}')

# Fine-tuned classification head
class ClassificationHead(tf.keras.Model):
    def __init__(self, num_classes, hidden_dim=512, dropout=0.0):
        super().__init__()
        self.fc1     = tf.keras.layers.Dense(hidden_dim)
        self.act     = tf.keras.layers.Activation('relu')
        self.dropout = tf.keras.layers.Dropout(dropout)
        self.fc2     = tf.keras.layers.Dense(num_classes)
    def call(self, x, training=False):
        x = self.fc1(x); x = self.act(x); x = self.dropout(x, training=training)
        return self.fc2(x)

head = ClassificationHead(NUM_CLASSES, HIDDEN_DIM, DROPOUT)
_ = head(tf.zeros((1, EMB_DIM)), training=False)

with h5py.File(WEIGHTS_PATH, 'r') as wf:
    head.fc1.kernel.assign(wf['fc1']['vars']['0'][:])
    head.fc1.bias.assign(  wf['fc1']['vars']['1'][:])
    head.fc2.kernel.assign(wf['fc2']['vars']['0'][:])
    head.fc2.bias.assign(  wf['fc2']['vars']['1'][:])
print(f'Fine-tuned weights loaded ← {WEIGHTS_PATH}')

## Inference

In [ ]:
@tf.function(input_signature=[tf.TensorSpec([None, CLIP_SAMPLES], tf.float32)])
def predict_both(clips):
    out    = sig(inputs=clips)
    emb    = tf.stop_gradient(out[EMB_KEY])
    # Perch zero-shot: pad label output so out-of-vocab index → 0
    label  = tf.pad(out['label'], [[0,0],[0,1]])          # (N, n_perch+1)
    perch_scores = tf.gather(label, perch_indices, axis=1) # (N, 234)
    # Fine-tuned head
    ft_logits = head(emb, training=False)
    ft_probs  = tf.sigmoid(ft_logits)
    return perch_scores, ft_probs


def process_soundscape(filepath):
    ss_id  = re.sub(r'\.ogg$', '', os.path.basename(filepath), flags=re.IGNORECASE)
    audio, _ = librosa.load(filepath, sr=SAMPLE_RATE, mono=True)
    audio  = audio.astype(np.float32)
    n_segs = len(audio) // CLIP_SAMPLES
    if n_segs == 0:
        return [], np.empty((0, NUM_CLASSES))

    clips   = np.stack([audio[i*CLIP_SAMPLES:(i+1)*CLIP_SAMPLES] for i in range(n_segs)])
    row_ids = [f'{ss_id}_{(i+1)*CLIP_DURATION}' for i in range(n_segs)]

    all_perch, all_ft = [], []
    for i in range(0, len(clips), BATCH_SIZE):
        batch = tf.constant(clips[i:i+BATCH_SIZE], dtype=tf.float32)
        p_scores, ft_probs = predict_both(batch)
        all_perch.append(p_scores.numpy())
        all_ft.append(ft_probs.numpy())

    perch_arr = np.concatenate(all_perch, axis=0)   # (N, 234) raw logits
    ft_arr    = np.concatenate(all_ft,    axis=0)   # (N, 234) sigmoid probs

    # Sigmoid-normalise Perch logits to [0,1]
    perch_probs = 1.0 / (1.0 + np.exp(-perch_arr))

    # Weighted blend
    blended = ALPHA * perch_probs + (1.0 - ALPHA) * ft_arr
    return row_ids, blended


ogg_files = sorted(glob.glob(os.path.join(SC_DIR, '*.ogg')))
all_row_ids, all_preds = [], []

for k, fpath in enumerate(ogg_files):
    if time.time() > TERMINATE_TIME:
        print(f'Time limit reached at {k}/{len(ogg_files)}')
        break
    if k % 200 == 0:
        print(f'[{k+1}/{len(ogg_files)}] elapsed={( time.time()-START)/60:.1f}m')
    try:
        row_ids, preds = process_soundscape(fpath)
    except Exception as e:
        print(f'ERROR {os.path.basename(fpath)}: {e}'); continue
    if row_ids:
        all_row_ids.extend(row_ids)
        all_preds.append(preds)

print(f'Done: {len(all_row_ids)} rows in {(time.time()-START)/60:.1f}m')

## Build Submission

In [ ]:
predictions = np.concatenate(all_preds, axis=0)
submission  = pd.DataFrame(predictions, columns=primary_labels)
submission.insert(0, 'row_id', all_row_ids)
submission  = sample_sub[['row_id']].merge(submission, on='row_id', how='left').fillna(0.0)
submission.to_csv('submission.csv', index=False)
print(f'submission.csv: {submission.shape[0]} rows × {len(primary_labels)} species')
print(f'Elapsed: {(time.time()-START)/60:.1f} min')
submission.head(5)